# Chapter 2: LLMs and Evaluation Basics
Parts of an LLM app (single call → conversation → retrieval → tools → agent loop) and the basic ways to evaluate.

## Run in Google Colab or locally
Upload this notebook alone and run cells from top to bottom. Setup installs missing packages;
all required sample data and helper functions are included below.
For Gemini calls, add `GEMINI_API_KEY` in Colab's key-shaped **Secrets** panel and enable notebook access,
or enter your key at the hidden prompt. Local runs use the single repository-root `.env`.
Never paste a key into a code cell. API calls use your own quota; notebook 04 needs no API key.
Students: **File → Save a copy in Drive** for your own copy and session.
Generated files live in the current session; download any results you want to keep before it ends.


In [ ]:
# Run once per fresh Colab session. Local users can run uv sync at the repo root.
import importlib.util
import subprocess
import sys

packages = {"google.genai": "google-genai", "dotenv": "python-dotenv",
            "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "rank_bm25": "rank-bm25"}
def available(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False
missing = [package for module, package in packages.items() if not available(module)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [ ]:
"""Small helpers shared by all notebooks (Gemini calls + data loading)."""
import json
import os
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Colab uses its own Secrets; local runs find the root .env.
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is None:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / ".env").is_file():
            load_dotenv(directory / ".env", override=False)
            break

MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-001")

_client = None


def client():
    """Create the Gemini client once and reuse it."""
    global _client
    if _client is None:
        key = os.getenv("GEMINI_API_KEY")
        if not key and userdata is not None:
            try:
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                pass
        if not key:
            from getpass import getpass
            key = getpass("Enter your own Gemini API key (hidden): ").strip()
        if not key:
            raise RuntimeError("GEMINI_API_KEY is missing. Add it to Colab Secrets or the root .env.")
        _client = genai.Client(api_key=key)
    return _client



def _generate(contents, system=None, temperature=0.0, as_json=False, model=None, retries=5):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if as_json else None,
    )
    for attempt in range(retries):
        try:
            resp = client().models.generate_content(model=model or MODEL, contents=contents, config=config)
            return json.loads(resp.text) if as_json else resp.text
        except Exception as e:  # rate limits (free tier) or temporary errors
            if attempt == retries - 1:
                raise
            wait = 10 * (attempt + 1)
            print(f"[retry {attempt + 1}] {type(e).__name__}: waiting {wait}s")
            time.sleep(wait)


def ask(prompt, system=None, temperature=0.0, as_json=False, model=None):
    """One prompt in, one answer out. as_json=True returns a Python dict."""
    return _generate(prompt, system, temperature, as_json, model)


def chat(history, system=None, temperature=0.0, model=None):
    """history = [{"role": "user" | "assistant", "content": "..."}, ...]"""
    contents = [
        types.Content(role="model" if m["role"] == "assistant" else "user",
                      parts=[types.Part(text=m["content"])])
        for m in history
    ]
    return _generate(contents, system, temperature, False, model)


def embed(texts):
    """Turn a list of texts into a numpy matrix (one row per text)."""
    r = client().models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in r.embeddings])


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# All sample prompts and data are embedded in this notebook.
def load_data():
    return {'ch2': {'chat_demo': [{'role': 'user', 'content': 'My name is Zara and I am learning evals.'}, {'role': 'assistant', 'content': 'Nice to meet you, Zara! Evals are a great skill.'}, {'role': 'user', 'content': 'What is my name and what am I learning?'}], 'past_emails': ["I still can't log in to my account, the password reset email never arrives", 'How do I change the billing address on my invoice?', 'The app crashes whenever I open the camera', 'Please cancel my subscription before the next payment', "Can't sign in after the update, it says wrong password", "Where can I download last month's receipt?"], 'retrieval_queries': ['i still cant log in', 'reset my login', 'get my receipt'], 'prompt_template': {'role': 'You are a polite customer support assistant.', 'objective': 'Write a short reply to the customer email. Success means the customer knows the next step.', 'instructions': 'Do not promise refunds. Do not invent policies. Keep it under 60 words.', 'reasoning': 'Think about what the customer really needs before writing.', 'output_format': 'Plain text, 2-3 sentences.', 'examples': "Email: 'Where is my invoice?'\nReply: 'Hello! You can download invoices from Settings > Billing. Let me know if you need more help.'"}, 'template_email': 'Hi, I cannot reset my password and I need access today.', 'extraction': [{'email': "Let's meet on 12 March 2026 at the office.", 'expected': '2026-03-12'}, {'email': 'The deadline is April 3rd, 2026, please do not miss it.', 'expected': '2026-04-03'}, {'email': 'Invoice due on 1 January 2027.', 'expected': '2027-01-01'}, {'email': 'See you on the 25th of December 2026.', 'expected': '2026-12-25'}], 'summary_email': 'From: Ali Khan\n\nHi team, the client meeting moved to Thursday 3pm. Please send the revised deck before then and book the big meeting room.', 'summary_a': 'Sender: Ali Khan\n- Client meeting moved to Thursday 3pm\n- Send the revised deck before the meeting\n- Book the big meeting room', 'summary_b': 'Ali wrote an email to the whole team. In this email he says many things, among them that a certain meeting with a client has been moved to a different day, which is Thursday, at about three in the afternoon, and that people should also do things such as sending a deck and maybe booking a room.', 'orders': {'1042': {'status': 'Shipped', 'item_type': 'electronics'}, '1043': {'status': 'Delivered', 'item_type': 'clothing'}}, 'return_windows_days': {'electronics': 15, 'clothing': 30}, 'agent_question': 'My order 1042: what is its status, and how many days is the return window for that item?'}}


In [ ]:
import numpy as np

data = load_data()["ch2"]

## 1. Single call (one prompt in, one answer out)
`temperature=0` makes answers more repeatable.

In [ ]:
print(ask("Summarize in one line: " + data["summary_email"], system="You summarize emails.", temperature=0))

## 2. Conversation + message roles
The model has no memory. We **send the whole history every time**.
Roles: `system` (rules), `user` (request), `assistant` (model reply), `tool` (tool result, see section 7).

In [ ]:
history = data["chat_demo"]          # list of {"role": ..., "content": ...}
for m in history:
    print(f"{m['role']:10}: {m['content']}")

print("\nMODEL:", chat(history, system="You are a helpful assistant."))

## 3. Prompt template (parts joined with delimiters)

In [ ]:
parts = data["prompt_template"]       # role, objective, instructions, reasoning, output_format, examples

def build_prompt(email):
    sections = []
    for name, text in parts.items():
        sections.append(f"### {name.upper()}\n{text}")          # ### = delimiter
    sections.append(f"### CONTEXT (customer email)\n<email>\n{email}\n</email>")
    return "\n\n".join(sections)

prompt = build_prompt(data["template_email"])
print(prompt)
print("\n--- MODEL REPLY ---")
print(ask(prompt))

## 4. Retrieval type 1: Keyword search (BM25)
Finds documents that **share words** with the question.

In [ ]:
from rank_bm25 import BM25Okapi

docs = data["past_emails"]
bm25 = BM25Okapi([d.lower().split() for d in docs])

def bm25_rank(query):
    scores = bm25.get_scores(query.lower().split())
    return list(np.argsort(scores)[::-1])          # best doc index first

for q in data["retrieval_queries"]:
    best = bm25_rank(q)[0]
    print(f"{q!r:25} -> {docs[best]}")

## 5. Retrieval type 2: Embedding search
Text → numbers. Similar **meaning** → similar numbers (cosine similarity, -1 to 1).

In [ ]:
doc_vectors = embed(docs)

def embedding_rank(query):
    q = embed([query])[0]
    scores = [cosine(q, v) for v in doc_vectors]
    return list(np.argsort(scores)[::-1])

for q in data["retrieval_queries"]:
    best = embedding_rank(q)[0]
    print(f"{q!r:25} -> {docs[best]}")

## 6. Retrieval type 3: Hybrid search with RRF
RRF (Reciprocal Rank Fusion): a doc scores `1 / (60 + rank)` in each list; add the scores.

In [ ]:
def rrf(*rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

for q in data["retrieval_queries"]:
    best = rrf(bm25_rank(q), embedding_rank(q))[0]
    print(f"{q!r:25} -> {docs[best]}")

## 7. Tools + agent loop (ReAct style)
The model asks for a tool, we run it, send the result back, repeat. **Always cap the steps.**

In [ ]:
from google.genai import types

def get_order_status(order_id: str):
    return data["orders"].get(order_id, {"error": "order not found"})

def get_return_window_days(item_type: str):
    return data["return_windows_days"].get(item_type, "unknown")

TOOLS = {"get_order_status": get_order_status, "get_return_window_days": get_return_window_days}

tool_spec = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="get_order_status", description="Get status and item type of an order",
        parameters={"type": "OBJECT", "properties": {"order_id": {"type": "STRING"}}, "required": ["order_id"]}),
    types.FunctionDeclaration(
        name="get_return_window_days", description="Get return window in days for an item type",
        parameters={"type": "OBJECT", "properties": {"item_type": {"type": "STRING"}}, "required": ["item_type"]}),
])
config = types.GenerateContentConfig(tools=[tool_spec], temperature=0)

In [ ]:
contents = [types.Content(role="user", parts=[types.Part(text=data["agent_question"])])]

for step in range(5):                                    # maximum 5 steps
    resp = client().models.generate_content(model=MODEL, contents=contents, config=config)
    msg = resp.candidates[0].content
    calls = [p.function_call for p in msg.parts if p.function_call]

    if calls:                                            # model wants tools
        contents.append(msg)
        for call in calls:
            result = TOOLS[call.name](**call.args)
            print(f"step {step}: TOOL {call.name}({dict(call.args)}) -> {result}")
            contents.append(types.Content(role="user", parts=[
                types.Part.from_function_response(name=call.name, response={"result": result})]))
        continue                                         # ask the model again

    print(f"step {step}: FINAL ANSWER ->", resp.text)    # no tool call = done
    break

## 8. Evaluation setup A: Absolute, reference-based
We know the correct answer, so compare (exact match).

In [ ]:
correct = 0
for item in data["extraction"]:
    out = ask(item["email"], system="Extract the date. Reply ONLY as YYYY-MM-DD.").strip()
    ok = out == item["expected"]
    correct += ok
    print(f"{item['expected']} vs {out}  {'OK' if ok else 'WRONG'}")
print(f"Accuracy: {correct}/{len(data['extraction'])}")

## 9. Evaluation setup B: Absolute, reference-free
No correct answer. Check **rules**: starts with `Sender:` and has exactly 3 bullets.

In [ ]:
def check_rules(summary):
    lines = [l for l in summary.strip().splitlines() if l.strip()]
    starts_ok = lines[0].startswith("Sender:")
    bullets = [l for l in lines if l.strip().startswith(("-", "*", "•"))]
    return {"starts_with_sender": starts_ok, "exactly_3_bullets": len(bullets) == 3}

print("Summary A:", check_rules(data["summary_a"]))
print("Summary B:", check_rules(data["summary_b"]))

## 10. Evaluation setup C: Comparative (pairwise)
Ask a judge which summary is better. We run **both orders** (A,B then B,A) to catch position bias.

In [ ]:
def pairwise(email, first, second):
    prompt = (f"Email:\n{email}\n\nSummary A:\n{first}\n\nSummary B:\n{second}\n\n"
              'Which summary is better (accurate, short, clear)? Reply JSON: {"winner": "A" or "B"}')
    return ask(prompt, as_json=True)["winner"]

r1 = pairwise(data["summary_email"], data["summary_a"], data["summary_b"])     # a first
r2 = pairwise(data["summary_email"], data["summary_b"], data["summary_a"])     # b first
winner1 = "summary_a" if r1 == "A" else "summary_b"
winner2 = "summary_b" if r2 == "A" else "summary_a"
print("Order 1 winner:", winner1)
print("Order 2 winner:", winner2)
print("Consistent (no position bias):", winner1 == winner2)